In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
import subprocess
subprocess.run([
    "curl", "-L", "-o", "/kaggle/working/boss.zip",
    "http://dde.binghamton.edu/download/ImageDB/BOSSbase_1.01.zip"
])

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 1594M  100 1594M    0     0  8549k      0  0:03:10  0:03:10 --:--:-- 8041k


CompletedProcess(args=['curl', '-L', '-o', '/kaggle/working/boss.zip', 'http://dde.binghamton.edu/download/ImageDB/BOSSbase_1.01.zip'], returncode=0)

In [3]:
import zipfile
import os

with zipfile.ZipFile('/kaggle/working/boss.zip', 'r') as z:
    z.extractall('/kaggle/working/BOSSBase/')

In [4]:
# Verify
files = os.listdir('/kaggle/working/BOSSBase/BOSSbase_1.01')
print(f"Total files: {len(files)}")
print(f"Sample: {files[:5]}")

Total files: 10000
Sample: ['5199.pgm', '4559.pgm', '1534.pgm', '6579.pgm', '972.pgm']


In [5]:
import os
for root, dirs, files in os.walk('/kaggle/working/BOSSBase/'):
    print(root, f"→ {len(files)} files")
    break  # just top level firstx

/kaggle/working/BOSSBase/ → 0 files


In [6]:
!pip install pyarrow xgboost joblib PyWavelets -q

In [7]:
!pip install pyarrow scikit-image pywavelets tqdm -q

In [8]:
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm
from joblib import Parallel, delayed
from skimage.feature import graycomatrix, graycoprops
import pywt
import time

# ── Config ────────────────────────────────────────────────────────────────────
BOSSBASE_DIR  = Path('/kaggle/working/BOSSBase/BOSSbase_1.01')
OUTPUT_DIR    = Path('/kaggle/working/output')
OUTPUT_DIR.mkdir(exist_ok=True)
COVER_CSV     = OUTPUT_DIR / 'cover.csv'
CHECKPOINT    = OUTPUT_DIR / 'cover_done_ids.npy'
N_JOBS        = -1   # all cores
BATCH_SZ      = 100
_EPS          = 1e-10

# ── Histogram features (1278-dim) ─────────────────────────────────────────────
def raw_histogram(image):
    hist, _ = np.histogram(image.ravel(), bins=256, range=(0, 255))
    return (hist / hist.sum()).astype(np.float32)

def difference_histogram(image, direction):
    img = image.astype(np.int16)
    if direction == "h":
        diff = (img[:, 1:] - img[:, :-1]).ravel()
    else:
        diff = (img[1:, :] - img[:-1, :]).ravel()
    hist, _ = np.histogram(diff, bins=511, range=(-255, 255))
    total = hist.sum()
    return (hist / total).astype(np.float32) if total > 0 else hist.astype(np.float32)

# ── GLCM features (10-dim) ────────────────────────────────────────────────────
def extract_glcm(image):
    glcm = graycomatrix(image, distances=[1], angles=[0, np.pi/2],
                        levels=256, symmetric=True, normed=True)
    stats = ["contrast","dissimilarity","homogeneity","energy","correlation"]
    return np.concatenate([graycoprops(glcm, s).ravel() for s in stats]).astype(np.float32)

# ── Wavelet features (192-dim) ────────────────────────────────────────────────
_WAV_RANGES = {"LH1":(-50,50),"HL1":(-50,50),"HH1":(-50,50),
               "LH2":(-30,30),"HL2":(-30,30),"HH2":(-30,30)}
_WAV_ORDER  = ["LH1","HL1","HH1","LH2","HL2","HH2"]

def extract_wavelet(image):
    coeffs = pywt.wavedec2(image.astype(np.float32), 'haar', level=2)
    lh1,hl1,hh1 = coeffs[-1]
    lh2,hl2,hh2 = coeffs[-2]
    sm = {"LH1":lh1,"HL1":hl1,"HH1":hh1,"LH2":lh2,"HL2":hl2,"HH2":hh2}
    parts = []
    for k in _WAV_ORDER:
        lo,hi = _WAV_RANGES[k]
        h, _ = np.histogram(sm[k].ravel(), bins=32, range=(lo,hi))
        t = h.sum()
        parts.append((h/t).astype(np.float32) if t > 0 else h.astype(np.float32))
    return np.concatenate(parts)

# ── HRAI features (255-dim) ───────────────────────────────────────────────────
def extract_hrai(image):
    hist, _ = np.histogram(image.ravel(), bins=256, range=(0,255))
    H = (hist / (hist.sum() + _EPS)).astype(np.float64)
    even, odd = H[0::2], H[1::2]
    hrai1 = ((even - odd) / (even + odd + _EPS)).astype(np.float32)
    hrai2 = np.diff(hrai1).astype(np.float32)
    return np.concatenate([hrai1, hrai2])

# ── Combined 1735-dim extractor ───────────────────────────────────────────────
def extract_all(image):
    return np.concatenate([
        raw_histogram(image),                    # 256
        difference_histogram(image, "h"),        # 511
        difference_histogram(image, "v"),        # 511
        extract_glcm(image),                     #  10
        extract_wavelet(image),                  # 192
        extract_hrai(image),                     # 255
    ]).astype(np.float32)                        # = 1735

# ── Column names ──────────────────────────────────────────────────────────────
FEATURE_COLUMNS = (
    [f"h_raw_{i}"   for i in range(256)] +
    [f"h_diffh_{i}" for i in range(511)] +
    [f"h_diffv_{i}" for i in range(511)] +
    [f"glcm_{s}_{a}" for s in ["contrast","dissimilarity","homogeneity","energy","correlation"]
                     for a in ["0deg","90deg"]] +
    [f"wav_{sb}_bin{b:02d}" for sb in _WAV_ORDER for b in range(32)] +
    [f"hrai1_{k}" for k in range(128)] +
    [f"hrai2_{k}" for k in range(127)]
)

print(f"Feature columns: {len(FEATURE_COLUMNS)}")  # should print 1735

Feature columns: 1735


In [11]:
from imageio.v2 import imread
def load_pgm(path):
    try:
        img = imread(path)

        img = np.array(img).copy()

        EXPECTED_SIZE = 512 * 512

        if img.size != EXPECTED_SIZE:
            print(f"[SKIP] {path.name}: invalid size {img.size}")
            return None

        img = img.reshape((512, 512))

        return img

    except Exception as e:
        print(f"[ERROR] {path.name}: {e}")
        return None

In [12]:
def process_image(pgm_path: Path):
    try:
        img    = load_pgm(pgm_path)
        img_id = int(pgm_path.stem)
        feats  = extract_all(img)
        row    = {"image_id": img_id, "label": 0, "payload": 0.0}
        for col, val in zip(FEATURE_COLUMNS, feats):
            row[col] = val
        return row
    except Exception as e:
        print(f"[WARN] {pgm_path.name}: {e}")
        return None

# ── Load checkpoint ───────────────────────────────────────────────────────────
done = set(np.load(CHECKPOINT).tolist()) if CHECKPOINT.exists() else set()
pgm_files = sorted(BOSSBASE_DIR.glob("*.pgm"))
pgm_files = [p for p in pgm_files if int(p.stem) not in done]
print(f"To process: {len(pgm_files)} | Already done: {len(done)}")

# ── Batch loop ────────────────────────────────────────────────────────────────
start = time.time()

for batch_start in tqdm(range(0, len(pgm_files), BATCH_SZ), desc="Batches"):
    batch   = pgm_files[batch_start : batch_start + BATCH_SZ]
    results = Parallel(n_jobs=N_JOBS, backend="loky")(
        delayed(process_image)(p) for p in batch
    )
    rows = [r for r in results if r is not None]
    if not rows:
        continue

    batch_df = pd.DataFrame(rows)
    batch_df[FEATURE_COLUMNS] = batch_df[FEATURE_COLUMNS].astype(np.float32)

    # Write header only on first write, append after
    write_header = not COVER_CSV.exists()
    batch_df.to_csv(COVER_CSV, mode='a', header=write_header, index=False)

    for p in batch:
        done.add(int(p.stem))
    np.save(CHECKPOINT, np.array(sorted(done), dtype=np.int32))

elapsed = time.time() - start
print(f"\nDone in {elapsed/60:.1f} min → {COVER_CSV}")
print(f"Rows written: {pd.read_csv(COVER_CSV, usecols=['image_id']).shape[0]:,}")


To process: 10000 | Already done: 0


Batches:   0%|          | 0/100 [00:00<?, ?it/s]


Done in 6.4 min → /kaggle/working/output/cover.csv
Rows written: 10,000


In [13]:
cover_csv=pd.read_csv("/kaggle/working/output/cover.csv")

In [15]:
cover_csv.tail(10)

,image_id,label,payload,h_raw_0,h_raw_1,h_raw_2,h_raw_3,h_raw_4,h_raw_5,h_raw_6,...,hrai2_117,hrai2_118,hrai2_119,hrai2_120,hrai2_121,hrai2_122,hrai2_123,hrai2_124,hrai2_125,hrai2_126
9990,9990,0,0.0,0.000465,0.000446,0.001045,0.002327,0.004295,0.005409,0.006611,...,-0.075757,-0.033333,0.199999,0.454544,-0.121212,-0.333332,0.999991,-1.499988,0.166664,0.404762
9991,9991,0,0.0,0.000061,0.003010,0.016655,0.008987,0.017876,0.019928,0.019192,...,-0.321678,-0.109091,0.200000,-0.333332,0.175438,0.472180,-0.123809,0.194138,-1.103913,1.401490
9992,9992,0,0.0,0.000019,0.004505,0.005558,0.005062,0.004818,0.005489,0.005692,...,0.228570,0.199999,0.111111,-0.111111,-0.428570,1.095234,-0.523807,-0.476189,-0.095238,0.043956
9993,9993,0,0.0,0.000820,0.026974,0.022232,0.018208,0.016350,0.015831,0.016926,...,-0.117665,-0.064999,-0.164522,0.115758,0.054764,0.077699,-0.223302,0.057088,-0.239583,0.954898
9994,9994,0,0.0,0.000000,0.000164,0.002590,0.006775,0.012527,0.017487,0.021854,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
9995,9995,0,0.0,0.000107,0.000114,0.001671,0.004673,0.004715,0.010143,0.014290,...,0.132231,0.054435,-0.152184,-0.076500,0.140462,0.061218,-0.272707,0.121739,-0.214578,0.587358
9996,9996,0,0.0,0.002010,0.006321,0.005558,0.004864,0.003887,0.002560,0.002010,...,0.372671,-0.619047,0.380952,-0.269841,0.055555,-0.119048,0.447004,0.118710,-0.103529,-0.136470
9997,9997,0,0.0,0.007545,0.021576,0.017342,0.015163,0.018009,0.026707,0.031872,...,0.999974,-0.999974,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
9998,9998,0,0.0,0.000122,0.000275,0.001694,0.004799,0.009178,0.015095,0.019230,...,-0.285713,0.714283,0.249999,-0.849996,0.690906,0.575755,0.333327,-0.333327,-0.066667,-1.327269
9999,9999,0,0.0,0.000935,0.004070,0.007504,0.008537,0.010994,0.012608,0.014034,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [17]:
cover_csv.shape

(10000, 1738)